# PP1 TRUST-MPS + RECAP-MPS benchmark (Colab CPU)
Phase 3F-B implementation. Execute cells manually. Development only unless a separately approved, single-use audit is explicitly authorized. Synthetic demo-rule outcomes are not clinical validity.

In [ ]:
# 1. Fresh Google Colab CPU runtime: information only; no inference.
import os, platform, importlib.metadata, importlib.util, pathlib
assert pathlib.Path('/content').is_dir() and os.environ.get('COLAB_RELEASE_TAG')
assert not pathlib.Path('/dev/nvidia0').exists() and not os.environ.get('COLAB_TPU_ADDR')
print('Python:', platform.python_version())
print({p: importlib.metadata.version(p) if importlib.util.find_spec(p.replace('-', '_')) else 'missing' for p in ('numpy','scipy','sklearn','joblib')})

In [ ]:
# 2. Mount Google Drive. Source V3 data and frozen models remain read-only.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Restore the code-only ZIP into a fresh workspace; refuse unsafe paths and overwrites.
from pathlib import Path
import zipfile
BUNDLE = Path('/content/drive/MyDrive/PP1_3F_B_Code_Inputs.zip')
ROOT = Path('/content/PP1_3F_B_workspace')
assert BUNDLE.is_file() and not ROOT.exists()
with zipfile.ZipFile(BUNDLE) as z:
    assert z.testzip() is None
    names = z.namelist()
    assert len(names) == len(set(names))
    for item in z.infolist():
        name = item.filename
        assert not item.is_dir() and not name.startswith(('/', '\\')) and '\\' not in name
        assert all(part not in ('', '.', '..') for part in name.split('/')) and ':' not in name
        assert (item.external_attr >> 16) & 0o170000 not in (0o120000,)
    ROOT.mkdir()
    for item in z.infolist():
        target = ROOT / item.filename
        target.parent.mkdir(parents=True, exist_ok=True)
        with z.open(item) as source, target.open('xb') as sink:
            import shutil; shutil.copyfileobj(source, sink)
print('Restored', len(names), 'code/lock files to', ROOT)

In [ ]:
# 4. Runtime compatibility: inspect the frozen environment BEFORE installation/loading.
import json, platform
FROZEN = Path('/content/drive/MyDrive/PP1_V3_training_runs/PP1_V3_training_20261020T214151Z_961db068_final')
DATASET = Path('/content/drive/MyDrive/PP1_V3_runs/PP1_Synthetic_Motor_Dataset_V3_20261020T210904Z_b8c13141')
recorded = json.loads((FROZEN/'training/environment.json').read_text())
print('Recorded Python:', recorded['python'], 'Current:', platform.python_version())
print('Recorded packages:', recorded['packages'])
assert recorded['python'] == platform.python_version(), 'STOP: use a compatible fresh Colab runtime'

In [ ]:
# 5. Install ONLY in Colab; pin to recorded training packages. Restart if pip requests it.
import subprocess, sys
wanted = {k: recorded['packages'][k] for k in ('numpy','scipy','scikit-learn','xgboost','catboost','joblib')}
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *[f'{k}=={v}' for k,v in wanted.items()]], check=True)
print('Restart runtime if required, then rerun setup and integrity cells.')

In [ ]:
# 6. Fail-closed design, split, dependency and checkpoint hash preflight. No joblib load.
import sys, importlib.metadata
sys.path.insert(0, str(ROOT))
from motor_scoring_research.innovations.benchmark.locks import verify_design, verify_frozen_models
spec, parents = verify_design(ROOT)
assert len(parents) == 168
for name, version in wanted.items():
    assert importlib.metadata.version(name) == version, f'Package mismatch: {name}'
# Enter the SHA256 recorded by the prior trusted read-only preflight, never guess it.
EXPECTED_SELECTED_MODELS_SHA256 = ''
assert len(EXPECTED_SELECTED_MODELS_SHA256) == 64, 'STOP: supply trusted selected_models.json SHA256'
frozen_manifest = verify_frozen_models(FROZEN, EXPECTED_SELECTED_MODELS_SHA256)
print('PASS: design, 168-parent split, packages and frozen checkpoint hashes')

## Development generation and evaluation
The next cell is the **first inference/generation gate**. It defaults to FALSE. Only the 84 development parents are accessible through this cell.

In [ ]:
# 7. Separate explicit approval for development-only benchmark generation/evaluation.
APPROVE_DEVELOPMENT_RUN = False
assert APPROVE_DEVELOPMENT_RUN is True, 'STOP: development run is not approved'
from datetime import datetime, timezone
from motor_scoring_research.innovations.benchmark.runner import run
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT = Path('/content/drive/MyDrive/PP1_V3_innovation_benchmark_runs') / f'development_{stamp}'
OUT.parent.mkdir(parents=True, exist_ok=True)
assert not OUT.exists()
run(workspace=ROOT, dataset=DATASET, frozen_run=FROZEN, output=OUT,
    selected_models_sha256=EXPECTED_SELECTED_MODELS_SHA256, approved_development=True, split='development')
print('Development report:', OUT/'summary.json')

## Single-use innovation audit: sealed
Audit access requires a separately approved, frozen development/code decision lock, a fixed persistent Drive registry, and a fresh run. A claim is permanent even if execution fails. The audit cell deliberately remains disabled.

In [ ]:
# 8. Audit approval gate. DO NOT enable in Phase 3F-B.
APPROVE_SINGLE_USE_AUDIT = False
assert APPROVE_SINGLE_USE_AUDIT is True, 'STOP: audit set is sealed'
# Only after independent review: copy the signed/reviewed DEVELOPMENT_LOCK.json
# to the fixed registry and enter its SHA256 here. Never delete AUDIT_STARTED.json.
DEVELOPMENT_LOCK_SHA256 = ''
assert len(DEVELOPMENT_LOCK_SHA256) == 64
from datetime import datetime, timezone
from motor_scoring_research.innovations.benchmark.runner import run
REGISTRY = Path('/content/drive/MyDrive/PP1_V3_innovation_audit_registry')
assert REGISTRY.is_dir() and not (REGISTRY/'AUDIT_STARTED.json').exists()
audit_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
AUDIT_OUT = Path('/content/drive/MyDrive/PP1_V3_innovation_benchmark_runs') / f'audit_{audit_stamp}'
assert not AUDIT_OUT.exists()
run(workspace=ROOT, dataset=DATASET, frozen_run=FROZEN, output=AUDIT_OUT,
    selected_models_sha256=EXPECTED_SELECTED_MODELS_SHA256, split='audit',
    approved_audit=True, registry=REGISTRY, development_lock_sha256=DEVELOPMENT_LOCK_SHA256)